In [1]:
import weaviate

client = weaviate.connect_to_local(port=8090, grpc_port=50051)

client.is_ready()

True

In [5]:
import joblib
import pandas as pd 

bbc_data = joblib.load(r'C:\Users\khale\ML\RAG\rag-hybrid-architecture\data\bbc_data.joblib')

In [6]:
bbc_data[0].keys()

dict_keys(['title', 'pubDate', 'guid', 'link', 'description', 'article_content'])

In [7]:
print(len(bbc_data))

9973


In [8]:
bbc_data[0]

{'title': 'Justin Welby: Political leaders should treat opponents as human beings',
 'pubDate': Timestamp('2024-01-01 00:00:04'),
 'guid': 'https://www.bbc.co.uk/news/uk-67844356',
 'link': 'https://www.bbc.co.uk/news/uk-67844356?at_medium=RSS&at_campaign=KARANGA',
 'description': 'The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.',
 'article_content': 'Justin Welby speaks on BBC Radio 4\'s Today programme as part of a special show guest edited by Dame Emma Warmsley The Archbishop of Canterbury has urged politicians not to treat their opponents as enemies but fellow human beings. Speaking to the BBC, the Most Rev Justin Welby warned Britain\'s leaders to avoid divisive topics. But he said our capacity "to disagree deeply and not destructively" is cause for hope. Later, he will deliver a new year\'s message reflecting on global conflicts and his wishes for a "peaceful 2024". The archbishop\'s intervention came during an interview for BB

In [9]:
from chunking import chunk_data

chunk_objs = chunk_data(bbc_data)

print(len(chunk_objs))

26466


In [ ]:
CLOUDFLARE_URL = "..."

In [ ]:
from weaviate.classes.config import Configure, Property, DataType

if client.collections.exists("bbc_collection"):
    client.collections.delete("bbc_collection")

collection = client.collections.create(
    name='bbc_collection',

    vectorizer_config=Configure.Vectorizer.text2vec_ollama(
        api_endpoint=CLOUDFLARE_URL,  
        model="nomic-embed-text",
        vectorize_collection_name=False
    ),

    generative_config=Configure.Generative.ollama(
        api_endpoint=CLOUDFLARE_URL,
        model="qwen2.5"
    ),

    reranker_config=Configure.Reranker.transformers(),

    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="chunk", data_type=DataType.TEXT),
        Property(name="pubDate", data_type=DataType.DATE),
        Property(name="description", data_type=DataType.TEXT),
        Property(name="link", data_type=DataType.TEXT, skip_vectorization=True),
        Property(name="guid", data_type=DataType.TEXT, skip_vectorization=True),
        Property(name="chunk_index", data_type=DataType.INT),
    ]
)
print("✅ Collection created")

✅ Collection created


c:\Users\khale\anaconda3\envs\RAG\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


In [30]:
collection = client.collections.get("bbc_collection")
print(len(collection))

26466


In [10]:
from datetime import datetime, timezone
import pandas as pd 
import tqdm

with collection.batch.dynamic() as batch:
    for obj in tqdm.tqdm(chunk_objs): 
        
        raw_date = obj['pubDate']
        formatted_date = None
        
        if pd.notna(raw_date): 
            if isinstance(raw_date, str):
                dt = datetime.strptime(raw_date, "%Y-%m-%d %H:%M:%S")
                formatted_date = dt.replace(tzinfo=timezone.utc)
            elif isinstance(raw_date, pd.Timestamp):
                formatted_date = raw_date.to_pydatetime().replace(tzinfo=timezone.utc)


        batch.add_object(
            properties={
                "title": obj["title"],
                "chunk": obj["chunk"],
                "pubDate": formatted_date, 
                "description": obj["description"],
                "link": obj["link"],
                "guid": obj["guid"],
                "chunk_index": obj["chunk_index"]
            }
        )

if len(collection.batch.failed_objects) > 0:
    print(collection.batch.failed_objects[0])


  0%|          | 0/26466 [00:00<?, ?it/s]

100%|██████████| 26466/26466 [59:52<00:00,  7.37it/s] 


In [27]:
from weaviate.classes.query import Filter
def filter_by_metadata(metadata_property: str, 
                       values: list[str], 
                       collection: "weaviate.collections.collection.sync.Collection" , 
                       limit: int = 5) -> list:
    """
    Retrieves objects from a specified collection based on metadata filtering criteria.

    This function queries a collection within the specified client to fetch objects that match 
    certain metadata criteria. It uses a filter to find objects whose specified 'property' contains 
    any of the given 'values'. The number of objects retrieved is limited by the 'limit' parameter.

    Args:
    metadata_property (str): The name of the metadata property to filter on.
    values (List[str]): A list of values to be matched against the specified property.
    collection_name (weaviate.collections.collection.sync.Collection): The collection to query.
    limit (int, optional): The maximum number of objects to retrieve. Defaults to 5.

    Returns:
    List[Object]: A list of objects from the collection that match the filtering criteria.
    """
   
    response = collection.query.fetch_objects(limit=limit, filters=Filter.by_property(metadata_property).contains_any(values))
 
    response_objects = [x.properties for x in response.objects]
    
    return response_objects

In [28]:
res = filter_by_metadata('title', ['Taylor Swift'], collection, limit = 2)

In [29]:
for i in res:
    print(i["chunk"])
    print(i["chunk_index"])
    print()

The 2024 awards season kicked off in style at the Golden Globes - the first major red carpet event of the year. A four-month strike kept actors away from the red carpet for much of the second half of 2023, but Hollywood stars made up for lost time and dressed to impress. Oppenheimer and Succession sweep Golden Globes Globes host Jo Koy's jokes fall flat and other big moments The winners in full Compared to the Oscars, the Golden Globes are usually a more relaxed affair, often resulting in a more playful mood on the red carpet. Margot Robbie - star of one of the night's biggest films, Barbie - was certainly having fun with her outfit. Robbie wore a series of pink looks for last year's promotional tour but it seemed she saved the best for last, wearing a custom-made replica 1977 Superstar Barbie outfit by Armani consisting of a hot pink sequinned dress accessorised with a tulle stole. Taylor Swift arrived in colourful sequins, too - for her, a striking green gown. The singer, nominated f

In [23]:
from ollama import Client
ollama_client = Client(host=CLOUDFLARE_URL)

def query_rewriting(query):

    rewrite_prompt = f"""
    You are an expert search system optimizer. 
    Rewrite the following user query to be more descriptive and comprehensive for a semantic search database. 
    Include relevant financial synonyms and context.
    Return ONLY the rewritten query text, nothing else.

    Original query: {query}
    """


    llm_response = ollama_client.generate(model='qwen2.5', prompt=rewrite_prompt)
    return llm_response["response"]

In [24]:
user_query = "Tell me the economic situation of the US in 2024."

print("orginal prompt : \n", user_query)
print("llm prompt : \n",query_rewriting(user_query))

orginal prompt : 
 Tell me the economic situation of the US in 2024.
llm prompt : 
 What will be the economic conditions, financial landscape, and market performance of the United States in the year 2024, including key indicators such as GDP growth, unemployment rates, inflation, and stock market trends?


In [34]:
from weaviate.classes.query import MetadataQuery, Rerank

def hybrid_search(query, num_retrieval, num_retrieval_ranker, alpha=0.5):

    response = collection.query.hybrid(
        query=query, 
        alpha=alpha,
        limit=num_retrieval,   
        rerank=Rerank(
            prop="chunk",                                    
            query=query 
        ),
        return_metadata=MetadataQuery(score=True, explain_score=True)
    )

    rag_context = []
    
    for obj in response.objects:
        if obj.metadata.rerank_score < 0:
            continue
            
        if len(rag_context) == num_retrieval_ranker:
            break
            
        title = obj.properties['title']
        chunk = obj.properties['chunk']
        formatted_doc = f"Title: {title}\nText: {chunk}"
        
        rag_context.append(formatted_doc)
        
    final_context_string = "\n\n---\n\n".join(rag_context)
    
    return final_context_string

In [ ]:
print(hybrid_search(query=user_query, num_retrieval=10, num_retrieval_ranker=3, alpha=0.4))

In [ ]:
def full_pipeline(query,num_retrieval, num_retrieval_ranker, alpha ):
    improved_query = query_rewriting(query)
    final_prompt = f"""You are a smart and professional assistant. Answer the user's question based on the attached documents only.
    If the answer is not in the documents, say "I do not have enough information".

    Question: {query}

    Documents:
    {hybrid_search(query=improved_query, num_retrieval=num_retrieval, num_retrieval_ranker=num_retrieval_ranker, alpha=alpha)}
    """
    
    llm_response = ollama_client.chat(
        model='qwen2.5', 
        messages=[
            {'role': 'user', 'content': final_prompt}
        ]
    )
    return llm_response['message']['content']

In [61]:
user_query = "Tell me the economic situation of the US in 2023."
res = full_pipeline(query=user_query, num_retrieval=10, num_retrieval_ranker=3, alpha=0.4)

In [60]:
print(res)

I do not have enough information.

The provided document contains limited details about the economic conditions for the United States in 2024. It mentions that the Office for Budget Responsibility (OBR) expects borrowing to rise slightly in the next financial year before remaining broadly in line with previous forecasts, and that overall debt as a proportion of GDP is set to rise over the next four years. However, the document does not provide a detailed analysis of key financial indicators, trends, and forecasts for the United States in 2024. Therefore, I cannot provide a comprehensive economic analysis as requested.
